# ثلاثة أنظمة: بصري، نصي، ودمج ميزاتهما

خلية مستقلة واحدة، بلا متغيرات من دفاتر سابقة. فعّل GPU وInternet لتحميل ملفات tokenizer فقط إن لم تكن مخزنة. لا تثبّت أو تُحدث PyTorch عشوائيًا. أوقف أي تدريب آخر قبل التشغيل.

## ماذا ننشئ؟

1. **Visual:** MobileNetV3-Small بأوزان حقبتي المتانة التي تحققت منها، دون إعادة تدريب.
2. **Text:** BERT-Tiny القديم بأوزان text_full_seed42.pt، و[CLS]→إسقاط256→المصنّف الأصلي، دون إعادة تدريب.
3. **Fusion:** ميزات256 بصرية +256 نصية→رأس512→256→128→2. يتدرب رأس الدمج فقط، وتظل الأوزان وإحصاءاتBatchNorm في الفرعين ثابتة. هذا دمج ميزات برأس متعلّم، وليس تدريبًا طرفًا لطرف.

تعريفBERT-Tiny يعتمد على الإعداد الرسمي prajjwal1/bert-tiny: طبقتان، hidden128، attention heads2، intermediate512. تُفحص أشكال الأوزان وتُحمَّل كل الطبقات بـstrict=True. لا تُنزَّل أوزانBERT أوImageNet؛ تأتي من ملفيك. يُحمّل tokenizer ويحفظ ضمن مجلد التجربة. المصدر: https://huggingface.co/prajjwal1/bert-tiny/blob/main/config.json .

## الروابط المفقودة وحدود المقارنة

ملف المشروع القديم يذكر رابطًا افتراضيًا عند فشل فكQR: http://embedded-qr-payload.com . نحافظ على URL المحفوظ كما هو في الفرع النصي لتجنب تغيير تجهيزه دون توثيق. نتائج النصي على ok=0 تقيس استجابته للمدخل الافتراضي، وليست كشفًا دلاليًا لرابط مستخرج. تُفصل النتائج إلى all/url_available/url_missing مع أعداد كل مجموعة.

عندما يغيب الرابط (ok≠1 أو نصURLفارغ)، يستخدم نظامFusion **التنبؤ البصري مباشرة**. لا يدخل الرأس النصي في هذا القرار. يُدرّب رأس الدمج على صفوفTrain التي يتوفر رابطها فقط؛ تُستخرج الميزات لكلTrain، لكن الصفوف ذات الرابط المفقود لا تُحدث الرأس. تقييمFusion يشمل جميعValidation مع هذا المسار الاحتياطي، لذا يُسمّى النظام مركبًا مع fallback عند وصفه.

رأس الدمج يبدأ بنسخة من مصنّف النصي: مصفوفة الطبقة الأولى تأخذ الميزات النصية وتصفّر أعمدة البصري مبدئيًا. يُفحص تطابق مخرجاتالبداية مع النصي عند توفر الرابط. يتعلم بعد ذلك أوزان الميزات البصرية. إذا لم يتحسنValidation، تبقى أوزانالبداية أفضل خيار.

## مراحل التنفيذ والوقت

- التحقق من بصمتي الأوزان اللتين أرسلتهما، ونسخ احتياطية جديدة دون تعديل الأصل.
- التحقق من القائمة النظيفة وCSV الأصلي. لا إعادة تقسيم، ولا فكQR جديد، ولا تحميل صور من الإنترنت.
- استخراج ميزاتValidation أولًا وعرض المقارنة الأولية.
- استخراج ميزاتTrain مرة واحدة، ضمن ملفاتshards من4096صف، مع SHA256 وفحص القراءة. عند إعادة تشغيل الخلية تُستخدم الدفعات السليمة المكتملة. هذه مرحلة تحضير مكلفة نسبيًا وليست تدريب الرأس. إذا وجد shard تالف أو ناقص الإيصال، يتوقف لحمايته ولا يستبدله بصمت.
- بعد اكتمالالاستخراج: تدريب الرأس فقط، حتى5حقب وباتش1024، وتوقف بعدحقبتين دون تحسن Macro-F1 الكامل علىValidation. تدريبالرأس أسرع عادة من إعادة تدريب الفرعين، لكن لا ضمان لزمن محدد.
- حفظ اسم مستقل لكلcheckpoint، مع قراءةCRC وتطابقالأوزان وحالةالمُحسّن. الاستكمال منآخرحقبة مكتملة سليمة. نفس إعداداتOUT لازمة؛ لتجربة إعداداتمختلفة استخدم مجلدًا جديدًا.
- عند النهاية: تحميلالأوزانالمختارة وإثبات أن البصري والنصي لم تتغيرأوزانهما، ثم حفظ جدولValidation للأنظمةالثلاثة.

المجلدالجديد: `/kaggle/working/qr_three_systems_frozen_v1`.
حجمميزةfloat32لـTrain+Validation نحو1.76GB قبلالـlogits والملفاتالإضافية؛ استهلاكRAMأعلى أثناءالتجميع. تحقق من وجود نحو4GB مساحةقرصحرة وعدةGBRAM. صورQRلا تُنسخ. لا تُضمّن feature_cache في حزمةالمراجعين تلقائيًا.

## الاستنتاج العلمي

لا يوجد ضمان أن الدمج يتفوق على النصي. ما نختبره هو القيمة الإضافية للميزات البصرية فوق رابطمتاح، ودورالبصري عند غيابالرابط. اعرض نتائجالأنظمةالثلاثة علىالمجموعاتنفسها وبوضوح. الاستقلالبين النماذج هنا يخصأنظمةالتنبؤ؛ بعضالأوزانوالتدريبالسابق مشتركة، فيجب توثيقذلك.

الـtext checkpointالقديم لايحويdata_signature؛ قراءةالأوزانلا تثبت عيناتتدريبه. ملفالدوسيه يصف تدريبًا علىTrainبتقسيم مجمع، لكن ينبغي توثيق عضويةTrainالقديمة قبل الادعاء باختبار مستقل. هذه الخلية لا تقيّمTest. Test سبق استخدامه لتجاربالبصري؛ أي تقريرلاحق للأنظمةالثلاثة يذكرذلك.

هذا الدفتر لا يغيّر نتائجالبصريالموثقة سابقًا. عنداكتماله أرسل `validation_selected_comparison.csv` و`completion.json` أولًا. لا تُعدّل الإعدادات بناءًعلىTest.

فُحصت صياغةالكود وبعض منطقالمقارنةوالحفظ محليًا. لم يُشغّلBERTأوGPUأوcheckpointحقيقي منKaggle في بيئةالمساعد.

## تصحيح الاستكمال: مفاتيح JSON

يستخدم هذا الإصدار تمثيل JSON موحدًا عند مقارنة config.json فقط، لتجنب اعتبار مفاتيح BERT الرقمية مختلفة عن تمثيلها النصي في JSON. لا يُعاد إسناد identity، ولا تُغيّر بصمة cache_id أو هوية ملفات الاستكمال. لا تُحذف ميزات أو إعدادات قديمة. إذا وُجد اختلاف فعلي، يُطبع المسار والقيمة المحفوظة والحالية ويتوقف الكود لحماية التقدم. شغّل خلية الكود كاملة بعد إيقاف النسخة السابقة؛ لا تغيّر OUT.


In [ ]:
print('Three systems: VISUAL, TEXT, FUSION — preparation started.',flush=True)
import os,json,time,uuid,random,hashlib,shutil,zipfile,gc,math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm
from transformers import BertConfig,BertModel,AutoTokenizer
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,roc_auc_score,average_precision_score,confusion_matrix

VISUAL_PATH=Path('/kaggle/working/mobilenet_continue/robust_aug_phase2_v1/best_epoch_02_796c6ee005f6.pt')
TEXT_PATH=Path('/kaggle/working/runs/text_full_seed42.pt')
VISUAL_SHA='11f170e26ba4e96a8ceaba4cbac66d26d5b8829aad9f79a1e898377eda92dfab'
TEXT_SHA='63d4fbc8afcd15ee39df27ad9fc8eec966d365b8c9ee02cef4f99c8bc79a98a4'
ROOT=Path('/kaggle/working/qr_visual_fresh')
OUT=Path('/kaggle/working/qr_three_systems_frozen_v1')
SOURCE_CSV=Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
EXPECTED_SIGNATURE='d4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
TOKENIZER_ID='prajjwal1/bert-tiny'
TOKENIZER_LOCAL=None  # optional local tokenizer folder if not already cached
CFG=dict(seed=42,max_len=128,feature_batch_size=128,shard_rows=4096,workers=2,
         head_batch_size=1024,head_lr=1e-4,weight_decay=1e-5,label_smoothing=0.1,
         head_epochs=5,patience=2,clip=1.0,threshold=0.5,precision='FP32',
         feature_version=1,missing_url_policy='fusion_falls_back_to_visual',
         fusion_init='text_head; zero visual columns in first layer',
         training='freeze both source models; train fusion head on available-URL Train rows')
assert torch.cuda.is_available(),'Enable GPU first; CUDA must be True.'
DEVICE=torch.device('cuda:0')
torch.set_num_threads(1)
os.environ['TOKENIZERS_PARALLELISM']='false'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
OUT.mkdir(parents=True,exist_ok=True)


def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def atomic_json(value,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    with tmp.open('w',encoding='utf-8') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def signature(frame):
    h=hashlib.sha256()
    for row in tqdm(frame[['path','label','split','url','ok','sha256']].itertuples(index=False,name=None),
                    total=len(frame),desc='Verify membership',mininterval=2):
        h.update((json.dumps(row,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    return h.hexdigest()

print('[1/6] Verify original checkpoints...',flush=True)
assert VISUAL_PATH.is_file() and TEXT_PATH.is_file(),'Restore the two source checkpoints first.'
assert file_hash(VISUAL_PATH)==VISUAL_SHA and file_hash(TEXT_PATH)==TEXT_SHA
visual_checkpoint=torch.load(VISUAL_PATH,map_location='cpu',weights_only=False)
text_checkpoint=torch.load(TEXT_PATH,map_location='cpu',weights_only=False)
assert visual_checkpoint['epoch']==2 and visual_checkpoint['data_signature']==EXPECTED_SIGNATURE
assert text_checkpoint['cfg']['max_len']==128
for cp in [visual_checkpoint,text_checkpoint]:
    for key,value in cp['model_state'].items():
        assert torch.is_tensor(value),key
        if value.is_floating_point():assert torch.isfinite(value).all(),key
backup_dir=OUT/'source_backup';backup_dir.mkdir(exist_ok=True)
for name,path,sha in [('visual',VISUAL_PATH,VISUAL_SHA),('text',TEXT_PATH,TEXT_SHA)]:
    backup=backup_dir/(name+'.pt')
    if not backup.exists():
        tmp=backup.with_suffix('.writing');shutil.copy2(path,tmp)
        assert file_hash(tmp)==sha;os.replace(tmp,backup)
    assert file_hash(backup)==sha

print('[2/6] Verify saved Train/Validation membership...',flush=True)
audit=json.loads((ROOT/'audit.json').read_text())
assert audit['data_signature']==EXPECTED_SIGNATURE and audit.get('train_ready') is True
df=None
for path in [ROOT/'verified_mobilenet_inputs/splits_verified.parquet',
             ROOT/'verified_mobilenet_inputs/splits_verified.csv.gz',ROOT/'splits_clean.parquet']:
    if not path.is_file():continue
    try:
        candidate=pd.read_parquet(path) if path.suffix=='.parquet' else pd.read_csv(path,keep_default_na=False)
        assert len(candidate)==1005700 and candidate.path.is_unique
        assert candidate.label.isin([0,1]).all() and candidate.ok.isin([0,1]).all()
        assert signature(candidate)==EXPECTED_SIGNATURE
        df=candidate;break
    except Exception as exc:print(path.name,type(exc).__name__,str(exc)[:150],flush=True)
assert df is not None,'Restore the matching audited manifest; do not create a new split.'
assert SOURCE_CSV.is_file() and file_hash(SOURCE_CSV)==audit['source_splits_sha256']
fields=['url','ok','label','domain','fold','split']
original=pd.read_csv(SOURCE_CSV,keep_default_na=False,usecols=['path']+fields)
assert original.path.is_unique and len(original)==audit['original_samples']
keys=df.source_path if 'source_path' in df else df.path
assert keys.is_unique
aligned=original.set_index('path').loc[keys]
for col in fields:
    if col in ['ok','label','fold']:
        assert np.array_equal(pd.to_numeric(df[col]).to_numpy(),pd.to_numeric(aligned[col]).to_numpy()),col
    else:
        assert np.array_equal(df[col].fillna('').astype(str).to_numpy(),aligned[col].fillna('').astype(str).to_numpy()),col
assert df.groupby('sha256').split.nunique().max()==1
assert df.groupby('sha256').label.nunique().max()==1
assert df.groupby('domain',dropna=False).split.nunique().max()==1
assert df[df.ok.eq(1)].groupby('url').split.nunique().max()==1
train=df[df.split.eq('train')].reset_index(drop=True)
val=df[df.split.eq('val')].reset_index(drop=True)
assert len(train)==710746 and len(val)==148529
for frame in [train,val]:
    # Preserve saved URL text exactly, including the legacy fallback URL.
    frame['url']=frame.url.fillna('').astype(str)
    frame['available']=frame.ok.eq(1)&frame.url.str.strip().ne('')
    assert frame.available.any()
print('Train:',len(train),'URL available:',int(train.available.sum()),flush=True)
print('Validation:',len(val),'URL available:',int(val.available.sum()),flush=True)
del df,candidate,original,aligned;gc.collect()

print('[3/6] Reconstruct both exact architectures + tokenizer...',flush=True)
class MobileNetVisual(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_backbone = torchvision.models.mobilenet_v3_small(weights=None)
        self.visual_backbone.classifier = nn.Identity()
        self.visual_proj = nn.Sequential(nn.Linear(576,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3))
        self.fusion = nn.Sequential(nn.Linear(256,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,2))
    def forward(self, x):
        return self.fusion(self.visual_proj(self.visual_backbone(x)))


def make_head(input_size):
    return nn.Sequential(nn.Linear(input_size,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.3),
                         nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Linear(128,2))

# Core tensor dimensions are inferred and checked. Attention-head count (2)
# is the official prajjwal1/bert-tiny config; it cannot be inferred from shapes alone.
text_state=text_checkpoint['model_state']
layer_ids={int(key.split('.')[3]) for key in text_state if key.startswith('text_backbone.encoder.layer.')}
# prefix text_backbone.encoder.layer.0... -> index is token 3
assert layer_ids=={0,1},f'Unexpected BERT layers: {layer_ids}'
assert text_state['text_backbone.embeddings.word_embeddings.weight'].shape==(30522,128)
assert text_state['text_backbone.encoder.layer.0.intermediate.dense.weight'].shape==(512,128)
bert_cfg=BertConfig(vocab_size=30522,hidden_size=128,num_hidden_layers=2,num_attention_heads=2,
    intermediate_size=512,hidden_act='gelu',hidden_dropout_prob=.1,attention_probs_dropout_prob=.1,
    max_position_embeddings=512,type_vocab_size=2,initializer_range=.02,layer_norm_eps=1e-12,pad_token_id=0)
bert_cfg._attn_implementation='eager'
HAS_POOLER=any(k.startswith('text_backbone.pooler.') for k in text_state)

class TextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_backbone=BertModel(bert_cfg,add_pooling_layer=HAS_POOLER)
        self.text_proj=nn.Sequential(nn.Linear(128,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.3))
        self.fusion=make_head(256)
    def features(self,tokens):
        outputs=self.text_backbone(**tokens,return_dict=True)
        return self.text_proj(outputs.last_hidden_state[:,0])
    def forward(self,tokens):return self.fusion(self.features(tokens))

class ThreeSystems(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual=MobileNetVisual();self.text=TextModel();self.fusion=make_head(512)
        for branch in [self.visual,self.text]:
            for parameter in branch.parameters():parameter.requires_grad_(False)
    def train(self,mode=True):
        super().train(mode);self.visual.eval();self.text.eval();return self
    def encode(self,images,tokens):
        with torch.no_grad():
            v=self.visual.visual_proj(self.visual.visual_backbone(images))
            t=self.text.features(tokens)
            return torch.cat([v,t],dim=1),self.visual.fusion(v),self.text.fusion(t)
    def forward(self,images,tokens,available):
        z,v_logits,t_logits=self.encode(images,tokens)
        fused=self.fusion(z)
        fused=torch.where(available[:,None],fused,v_logits)
        return dict(visual=v_logits,text=t_logits,fusion=fused)

random.seed(CFG['seed']);np.random.seed(CFG['seed']);torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
model=ThreeSystems()
model.visual.load_state_dict(visual_checkpoint['model_state'],strict=True)
model.text.load_state_dict(text_state,strict=True)
# Warm-start: initial fusion matches the text classifier whenever a URL exists.
head_state={k:v.clone() for k,v in model.text.fusion.state_dict().items()}
head_state['0.weight']=torch.cat([torch.zeros_like(head_state['0.weight']),head_state['0.weight']],dim=1)
model.fusion.load_state_dict(head_state,strict=True)
model.to(DEVICE).eval()
del visual_checkpoint,text_checkpoint,text_state,head_state;gc.collect()

tokenizer_dir=OUT/'tokenizer'
try:
    if tokenizer_dir.is_dir():tokenizer=AutoTokenizer.from_pretrained(str(tokenizer_dir),use_fast=True,local_files_only=True)
    else:
        tokenizer=AutoTokenizer.from_pretrained(TOKENIZER_LOCAL or TOKENIZER_ID,use_fast=True)
        tokenizer.save_pretrained(tokenizer_dir)
except Exception as exc:
    raise RuntimeError('Tokenizer unavailable: enable Kaggle Internet or set TOKENIZER_LOCAL to a saved tokenizer folder. No model-weight download is needed.') from exc
assert len(tokenizer)==30522 and tokenizer.cls_token_id==101 and tokenizer.sep_token_id==102 and tokenizer.pad_token_id==0
vocab_fingerprint=hashlib.sha256(json.dumps(tokenizer.get_vocab(),sort_keys=True,separators=(',',':')).encode()).hexdigest()
identity=dict(cfg=CFG,data_signature=EXPECTED_SIGNATURE,visual_sha256=VISUAL_SHA,text_sha256=TEXT_SHA,
    bert_config=bert_cfg.to_dict(),has_pooler=HAS_POOLER,tokenizer_id=TOKENIZER_ID,vocab_sha256=vocab_fingerprint,
    tokenizer_special_ids=dict(cls=101,sep=102,pad=0),
    tokenizer_files_sha256={p.name:file_hash(p) for p in tokenizer_dir.iterdir() if p.is_file()},tokenizer_lower_case=getattr(tokenizer,'do_lower_case',None),
    versions=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__)))
# JSON converts integer dictionary keys to strings (e.g. BERT id2label).
# Normalize only the comparison; preserve the ORIGINAL identity object for
# existing feature-cache hashes and torch checkpoint compatibility.
def normalize_json(value):
    return json.loads(json.dumps(value,ensure_ascii=False,allow_nan=False))

def identity_differences(old,current,prefix=''):
    differences=[]
    if isinstance(old,dict) and isinstance(current,dict):
        for key in sorted(set(old)|set(current)):
            name=f'{prefix}.{key}' if prefix else key
            if key not in old:differences.append((name,'<missing>',current[key]))
            elif key not in current:differences.append((name,old[key],'<missing>'))
            else:differences.extend(identity_differences(old[key],current[key],name))
    elif old!=current:
        differences.append((prefix,old,current))
    return differences

config_path=OUT/'config.json'
if config_path.exists():
    saved_config=json.loads(config_path.read_text())
    current_config=normalize_json(identity)
    differences=identity_differences(saved_config,current_config)
    if differences:
        print('Actual configuration differences (saved -> current):',flush=True)
        for name,old_value,new_value in differences[:40]:
            print(name,repr(old_value)[:300],'->',repr(new_value)[:300],flush=True)
        raise ValueError('Run settings really differ. Keep existing features; send the differences above. Do not bypass this check.')
    print('PASS: JSON-normalized settings match; cached features can be checked and resumed.',flush=True)
else:
    atomic_json(identity,config_path)

RESIZE=T.Resize((224,224))
NORMALIZE=T.Compose([T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])
class JointDataset(Dataset):
    def __init__(self,frame):
        self.paths=frame.path.to_numpy();self.urls=frame.url.to_numpy();self.labels=frame.label.to_numpy(dtype=np.int64)
    def __len__(self):return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:x=NORMALIZE(RESIZE(im.convert('RGB')))
        return x,self.urls[i],int(self.labels[i])

print('[4/6] Resumable frozen-feature extraction — not head training yet...',flush=True)
cache=OUT/'feature_cache';cache.mkdir(exist_ok=True)
cache_id=hashlib.sha256(json.dumps(identity,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def shard_valid(path,frame,start,end):
    receipt_path=path.with_suffix('.json')
    assert path.is_file() and receipt_path.is_file()
    receipt=json.loads(receipt_path.read_text())
    assert receipt['cache_id']==cache_id and receipt['start']==start and receipt['end']==end
    assert file_hash(path)==receipt['sha256']
    with np.load(path,allow_pickle=False) as data:
        assert set(data.files)=={'features','visual_logits','text_logits','labels','available'}
        assert data['features'].shape==(end-start,512)
        assert data['visual_logits'].shape==data['text_logits'].shape==(end-start,2)
        assert np.array_equal(data['labels'],frame.label.iloc[start:end].to_numpy())
        assert np.array_equal(data['available'],frame.available.iloc[start:end].to_numpy())
        for key in ['features','visual_logits','text_logits']:assert np.isfinite(data[key]).all()


def extract_features(frame,split):
    paths=[];model.eval()
    bar=tqdm(total=len(frame),desc=split+' cached features',mininterval=2)
    try:
        for start in range(0,len(frame),CFG['shard_rows']):
            end=min(start+CFG['shard_rows'],len(frame));path=cache/f'{split}_{start:07d}_{end:07d}.npz'
            if path.exists() or path.with_suffix('.json').exists():
                shard_valid(path,frame,start,end)
                paths.append(path);bar.update(end-start);continue
            feature_parts=[];visual_parts=[];text_parts=[]
            loader=DataLoader(JointDataset(frame.iloc[start:end]),batch_size=CFG['feature_batch_size'],
                shuffle=False,num_workers=CFG['workers'],pin_memory=True,prefetch_factor=2 if CFG['workers'] else None)
            with torch.no_grad():
                for images,urls,labels in loader:
                    tokens=tokenizer(list(urls),padding='max_length',truncation=True,
                        max_length=CFG['max_len'],return_tensors='pt')
                    tokens={k:v.to(DEVICE,non_blocking=True) for k,v in tokens.items()}
                    z,v,t=model.encode(images.to(DEVICE,non_blocking=True),tokens)
                    feature_parts.append(z.float().cpu().numpy());visual_parts.append(v.float().cpu().numpy());text_parts.append(t.float().cpu().numpy())
                    bar.update(len(labels))
            payload=dict(features=np.concatenate(feature_parts),visual_logits=np.concatenate(visual_parts),
                text_logits=np.concatenate(text_parts),labels=frame.label.iloc[start:end].to_numpy(dtype=np.int64),
                available=frame.available.iloc[start:end].to_numpy(dtype=bool))
            for key in ['features','visual_logits','text_logits']:assert np.isfinite(payload[key]).all()
            # A new immutable shard. Existing shards are never silently replaced.
            tmp=path.with_name(path.name+'.writing_'+uuid.uuid4().hex[:8])
            with tmp.open('xb') as f:np.savez(f,**payload);f.flush();os.fsync(f.fileno())
            with np.load(tmp,allow_pickle=False) as checked:
                for key,value in payload.items():assert np.array_equal(checked[key],value)
            sha=file_hash(tmp);os.replace(tmp,path)
            atomic_json(dict(cache_id=cache_id,start=start,end=end,sha256=sha),path.with_suffix('.json'))
            paths.append(path)
            print(f'{split}: verified {end:,}/{len(frame):,} rows ({end/len(frame):.1%})',flush=True)
            del payload,feature_parts,visual_parts,text_parts
    finally:bar.close()
    return paths

# Validate before expensive Train extraction; compare the three initial systems.
val_paths=extract_features(val,'val')

def load_features(paths,frame):
    n=len(frame);arrays=dict(features=np.empty((n,512),dtype=np.float32),
        visual_logits=np.empty((n,2),dtype=np.float32),text_logits=np.empty((n,2),dtype=np.float32),
        labels=frame.label.to_numpy(dtype=np.int64),available=frame.available.to_numpy(dtype=bool))
    pos=0
    for path in tqdm(paths,desc='Load verified feature shards',mininterval=2):
        with np.load(path,allow_pickle=False) as shard:
            count=len(shard['labels'])
            for key in ['features','visual_logits','text_logits']:arrays[key][pos:pos+count]=shard[key]
            assert np.array_equal(arrays['labels'][pos:pos+count],shard['labels'])
            assert np.array_equal(arrays['available'][pos:pos+count],shard['available'])
        pos+=count
    assert pos==n
    return arrays

val_data=load_features(val_paths,val)

def metrics(y,p):
    if len(y)==0:return dict(samples=0)
    pred=(p>=CFG['threshold']).astype(np.int64)
    return dict(samples=int(len(y)),accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,average='macro',zero_division=0)),
        malicious_precision=float(precision_score(y,pred,zero_division=0)),malicious_recall=float(recall_score(y,pred,zero_division=0)),
        malicious_f1=float(f1_score(y,pred,zero_division=0)),
        roc_auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,
        average_precision=float(average_precision_score(y,p)) if len(np.unique(y))==2 else None,
        confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist())


def evaluate_three(data,tag):
    model.eval();parts=[]
    with torch.no_grad():
        for start in range(0,len(data['labels']),4096):
            z=torch.from_numpy(data['features'][start:start+4096]).to(DEVICE)
            parts.append(model.fusion(z).cpu())
    fused=torch.cat(parts);mask=torch.from_numpy(data['available'])
    visual=torch.from_numpy(data['visual_logits']);text=torch.from_numpy(data['text_logits'])
    fused=torch.where(mask[:,None],fused,visual)
    if tag=='initial':
        assert torch.allclose(fused[mask],text[mask],atol=1e-4,rtol=1e-4),'Fusion init should match text logits on available URLs.'
        assert torch.equal(fused[~mask],visual[~mask])
    rows=[];predictions=val[['path','label','ok','available']].copy()
    for name,logits in [('visual',visual),('text',text),('fusion',fused)]:
        p=logits.softmax(1)[:,1].numpy();assert np.isfinite(p).all()
        predictions[name+'_probability']=p
        for group,selection in [('all',np.ones(len(p),dtype=bool)),('url_available',data['available']),('url_missing',~data['available'])]:
            rows.append(dict(system=name,group=group,**metrics(data['labels'][selection],p[selection])))
    table=pd.DataFrame(rows);table.drop(columns=['confusion_matrix'],errors='ignore').to_csv(OUT/f'validation_{tag}_comparison.csv',index=False)
    predictions.to_csv(OUT/f'validation_{tag}_predictions.csv.gz',index=False,compression='gzip')
    for row in rows:
        if row['group']=='all':print(f"{tag} {row['system']}: Acc={row['accuracy']:.4%} | Macro-F1={row['macro_f1']:.4%} | AUC={row['roc_auc']:.5f}",flush=True)
    atomic_json(rows,OUT/f'validation_{tag}_metrics.json')
    return next(row for row in rows if row['system']=='fusion' and row['group']=='all')

initial=evaluate_three(val_data,'initial')
print('Text missing-URL results use the legacy saved URL; fusion missing-URL results use visual fallback.',flush=True)
train_paths=extract_features(train,'train')
train_data=load_features(train_paths,train)
print('[5/6] Feature extraction complete. Training ONLY the fusion head...',flush=True)

# Most memory is frozen feature arrays (~1.8 GB CPU for both splits), not images.
# Head inputs are moved per batch. Both source networks stay unchanged/eval.
optimizer=torch.optim.AdamW(model.fusion.parameters(),lr=CFG['head_lr'],weight_decay=CFG['weight_decay'])
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG['head_epochs'],eta_min=1e-6)
criterion=nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

def cpu_state():return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def verified_save(payload,label):
    path=OUT/(label+'_'+uuid.uuid4().hex[:12]+'.pt');tmp=path.with_suffix('.writing')
    print('Saving and verifying:',path.name,flush=True)
    with tmp.open('xb') as f:torch.save(payload,f);f.flush();os.fsync(f.fileno())
    with zipfile.ZipFile(tmp) as archive:assert archive.testzip() is None
    restored=torch.load(tmp,map_location='cpu',weights_only=False)
    assert set(restored['model_state'])==set(payload['model_state'])
    for key,value in payload['model_state'].items():assert torch.equal(restored['model_state'][key],value.cpu()),key
    if 'optimizer' in payload:
        assert restored['optimizer']['param_groups']==payload['optimizer']['param_groups']
        for index,entry in payload['optimizer']['state'].items():
            for key,value in entry.items():
                saved=restored['optimizer']['state'][index][key]
                if torch.is_tensor(value):assert torch.equal(saved,value.cpu())
                else:assert saved==value
    sha=file_hash(tmp);os.replace(tmp,path)
    atomic_json(dict(file=path.name,sha256=sha,epoch=payload['epoch']),path.with_suffix('.json'))
    print('PASS:',path.name,flush=True)
    return path


def checkpoint_payload(epoch,best,best_file,stale,history,full=True):
    payload=dict(model_state=cpu_state(),epoch=epoch,cfg=CFG,identity=identity,data_signature=EXPECTED_SIGNATURE,
                 visual_sha256=VISUAL_SHA,text_sha256=TEXT_SHA,validation_history=history)
    if full:payload.update(optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),best_macro_f1=best,
        best_file=best_file,stale=stale,history=history,python_rng=random.getstate(),numpy_rng=np.random.get_state(),
        torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all())
    return payload

start_epoch=0;best=initial['macro_f1'];best_file=None;stale=0;history=[dict(epoch=0,validation=initial)]
receipts=[]
for receipt_path in OUT.glob('epoch_*.json'):
    receipt=json.loads(receipt_path.read_text());receipts.append((int(receipt['epoch']),receipt))
if receipts:
    resumed=False
    for _,receipt in sorted(receipts,key=lambda r:r[0],reverse=True):
        try:
            path=OUT/receipt['file'];assert file_hash(path)==receipt['sha256']
            cp=torch.load(path,map_location='cpu',weights_only=False)
            assert cp['identity']==identity
            best_path=OUT/cp['best_file'];best_receipt=json.loads(best_path.with_suffix('.json').read_text())
            assert file_hash(best_path)==best_receipt['sha256']
        except Exception as exc:print('Cannot resume',receipt['file'],type(exc).__name__,flush=True);continue
        model.load_state_dict(cp['model_state'],strict=True);optimizer.load_state_dict(cp['optimizer']);scheduler.load_state_dict(cp['scheduler'])
        start_epoch=cp['epoch'];best=cp['best_macro_f1'];best_file=cp['best_file'];stale=cp['stale'];history=cp['history']
        random.setstate(cp['python_rng']);np.random.set_state(cp['numpy_rng']);torch.set_rng_state(cp['torch_rng']);torch.cuda.set_rng_state_all(cp['cuda_rng'])
        del cp;resumed=True;print('Resumed fusion head after epoch',start_epoch,flush=True);break
    assert resumed,'No valid full fusion checkpoint. Source files remain unchanged.'
else:
    # Real optimizer step to test a full save. Restore before actual training.
    initial_head={k:v.detach().cpu().clone() for k,v in model.fusion.state_dict().items()}
    rng_py=random.getstate();rng_np=np.random.get_state();rng_cpu=torch.get_rng_state();rng_cuda=torch.cuda.get_rng_state_all()
    indices=np.flatnonzero(train_data['available'])[:32];assert len(indices)>1
    model.train();optimizer.zero_grad(set_to_none=True)
    x=torch.from_numpy(train_data['features'][indices]).to(DEVICE);y=torch.from_numpy(train_data['labels'][indices]).to(DEVICE)
    loss=criterion(model.fusion(x),y);assert torch.isfinite(loss);loss.backward()
    nn.utils.clip_grad_norm_(model.fusion.parameters(),CFG['clip'],error_if_nonfinite=True)
    optimizer.step()
    verified_save(checkpoint_payload(-1,best,None,0,history),'storage_smoke')
    model.fusion.load_state_dict(initial_head,strict=True);optimizer.state.clear();optimizer.zero_grad(set_to_none=True)
    random.setstate(rng_py);np.random.set_state(rng_np);torch.set_rng_state(rng_cpu);torch.cuda.set_rng_state_all(rng_cuda)
    del initial_head,x,y,loss
    best_file=verified_save(checkpoint_payload(0,best,None,0,history,full=False),'best_epoch_00').name
    verified_save(checkpoint_payload(0,best,best_file,0,history),'epoch_00')
    atomic_json(history,OUT/'history.json')

eligible_indices=np.flatnonzero(train_data['available'])
for epoch in range(start_epoch+1,CFG['head_epochs']+1):
    if stale>=CFG['patience']:break
    model.train();order=np.random.permutation(eligible_indices);seen=0;correct=0;total_loss=0;started=time.time()
    # Avoid a final singleton with BatchNorm; merge it into the preceding batch.
    slices=[order[start:start+CFG['head_batch_size']] for start in range(0,len(order),CFG['head_batch_size'])]
    if len(slices)>1 and len(slices[-1])==1:
        final_single=slices.pop()
        slices[-1]=np.concatenate([slices[-1],final_single])
    bar=tqdm(slices,desc=f'Fusion head {epoch}/{CFG["head_epochs"]}',mininterval=1)
    for batch,indices in enumerate(bar,1):
        x=torch.from_numpy(train_data['features'][indices]).to(DEVICE)
        y=torch.from_numpy(train_data['labels'][indices]).to(DEVICE)
        optimizer.zero_grad(set_to_none=True);logits=model.fusion(x);loss=criterion(logits,y)
        assert torch.isfinite(loss);loss.backward()
        nn.utils.clip_grad_norm_(model.fusion.parameters(),CFG['clip'],error_if_nonfinite=True);optimizer.step()
        seen+=len(indices);correct+=int((logits.detach().argmax(1)==y).sum());total_loss+=float(loss.detach())*len(indices)
        if batch==1 or batch%50==0 or batch==len(slices):
            speed=seen/max(time.time()-started,1e-6)
            print(f'Fusion HEAD epoch {epoch}: {seen:,}/{len(order):,} ({seen/len(order):.1%}) | loss={total_loss/seen:.4f} | acc={correct/seen:.2%} | {speed:.0f} rows/s',flush=True)
    assert seen==len(eligible_indices)
    score=evaluate_three(val_data,f'epoch_{epoch:02d}');improved=score['macro_f1']>best+1e-6
    history.append(dict(epoch=epoch,train_loss=total_loss/seen,validation=score,improved=improved))
    if improved:
        best=score['macro_f1'];stale=0
        best_file=verified_save(checkpoint_payload(epoch,best,None,0,history,full=False),f'best_epoch_{epoch:02d}').name
    else:stale+=1
    scheduler.step()
    verified_save(checkpoint_payload(epoch,best,best_file,stale,history),f'epoch_{epoch:02d}')
    atomic_json(history,OUT/'history.json')
    print(f'Verified epoch {epoch}. Best full Validation fusion Macro-F1={best:.4%}',flush=True)

print('[6/6] Restore selected fusion weights and report all three systems...',flush=True)
best_path=OUT/best_file;receipt=json.loads(best_path.with_suffix('.json').read_text())
assert file_hash(best_path)==receipt['sha256']
selected=torch.load(best_path,map_location='cpu',weights_only=False);model.load_state_dict(selected['model_state'],strict=True)
# Prove the old visual and text systems were not changed during head training.
for source_name,branch in [('visual',model.visual),('text',model.text)]:
    source_cp=torch.load(backup_dir/(source_name+'.pt'),map_location='cpu',weights_only=False)
    assert set(branch.state_dict())==set(source_cp['model_state'])
    for key,value in branch.state_dict().items():assert torch.equal(value.cpu(),source_cp['model_state'][key]),key
selected_score=evaluate_three(val_data,'selected')
assert abs(selected_score['macro_f1']-best)<1e-6
assert file_hash(VISUAL_PATH)==VISUAL_SHA and file_hash(TEXT_PATH)==TEXT_SHA
atomic_json(dict(selected_weights=str(best_path),best_validation_macro_f1=best,selected_epoch=selected['epoch'],
    cache_shards_train=len(train_paths),cache_shards_val=len(val_paths),
    source_models_unchanged=True,test_evaluated=False,
    legacy_text_provenance='No data_signature in original text checkpoint. Dossier describes grouped Train training; exact old membership must be documented before claiming independent Test performance.'),OUT/'completion.json')
print('Selected FUSION weights:',best_path,flush=True)
print('Validation comparison:',OUT/'validation_selected_comparison.csv',flush=True)
print('No Test images or Test labels were used for feature extraction, training or model selection.',flush=True)
print('Keep tokenizer/, config.json, selected .pt + receipt, history.json and source backups. Feature cache is reusable but large.',flush=True)
